# Sistema de Recomendação por Fatoração de Matriz (SVD)

Este notebook implementa um sistema de recomendação baseado em **Fatoração de Matriz via Decomposição em Valores Singulares** (SVD — *Singular Value Decomposition*).

## Como o Algoritmo Funciona

### O Problema

A matriz de ratings R (usuários × filmes) é altamente esparsa: a maioria dos usuários avaliou apenas uma fração ínfima do catálogo. Os métodos de vizinhança (User-Based e Item-Based) exploram essa esparsidade diretamente, mas dependem de ter vizinhos com filmes em comum.

A **Fatoração de Matriz** resolve o problema de forma diferente: em vez de comparar usuários ou itens diretamente, ela projeta usuários e filmes em um **espaço latente de baixa dimensão**. Nesse espaço, preferências similares ficam próximas.

### A Decomposição SVD

Qualquer matriz R ($m \times n$) pode ser decomposta exatamente como:

$$R = U \Sigma V^\top$$

Onde:
- **U** ($m \times m$): fatores latentes dos usuários — cada linha é a "assinatura" de um usuário no espaço latente.
- **$\Sigma$** ($m \times n$): matriz diagonal de **valores singulares** em ordem decrescente — indicam a importância de cada dimensão latente.
- **$V^\top$** ($n \times n$): fatores latentes dos filmes — cada coluna é a "assinatura" de um filme.

### SVD Truncado (Aproximação de Posto k)

Pelo **Teorema de Eckart-Young**, a melhor aproximação de posto $k$ da matriz R é:

$$R \approx \hat{R} = U_k \Sigma_k V_k^\top$$

onde $U_k$, $\Sigma_k$, $V_k^\top$ retêm apenas as $k$ maiores dimensões. Essa aproximação:
- **Remove o ruído** (dimensões pequenas capturam aleatoriedade, não padrões)
- **Generaliza** para pares (usuário, filme) não observados
- **Comprime** a representação: em vez de armazenar $m \times n$ entradas, armazenamos $(m + n) \times k$

O parâmetro $k$ é o principal hiperparâmetro — controla o trade-off entre **capacidade** (capturar padrões complexos) e **generalização** (não memorizar ruído).

### Fatores Latentes: o que representam?

O SVD descobre automaticamente dimensões de preferência. Embora não tenham nomes fixos, os fatores tendem a corresponder a conceitos como:
- Gênero cinematográfico (ação vs. drama)
- Tom emocional (leve vs. sombrio)
- Estilo visual (blockbuster vs. cinema de autor)

Filmes com vetores similares em $V_k$ são similares no espaço latente. Usuários com vetores similares em $U_k$ têm gostos parecidos.

### Imputação Baseline

O SVD requer uma matriz **completa** (sem NaN). Imputamos os valores ausentes com um **preditor de baseline** que considera viés de usuário e de item:

$$\hat{r}_{u,i}^{\text{baseline}} = \mu + b_u + b_i$$

Onde $\mu$ é a média global, $b_u$ é o viés do usuário $u$ (tende a dar notas altas ou baixas), e $b_i$ é o viés do item $i$ (filme muito bom ou muito ruim vs. média).

### Predição Final

Após a decomposição, a nota predita para o usuário $u$ no filme $i$ é simplesmente:

$$\hat{r}_{u,i} = \hat{R}[u, i] = \left(U_k \Sigma_k V_k^\top\right)[u, i]$$

Ao contrário dos métodos de vizinhança, **não há busca de vizinhos em tempo de inferência** — o custo é pago uma vez no SVD e a predição é $O(k)$ por par.

## MF-SVD vs Vizinhança

| Aspecto | User-Based / Item-Based | MF-SVD |
|---------|------------------------|--------|
| Representação | Esparsa (ratings brutos) | Densa (fatores latentes) |
| Inferência | Busca de vizinhos | Lookup em $\hat{R}$ |
| Generalização | Requer overlap de itens | Projeta no espaço latente |
| Interpretabilidade | Alta ("usuários como você") | Baixa (dimensões abstratas) |
| Cold start | Difícil | Difícil (requer SVD incremental) |
| Escalabilidade | Limitada | Boa (SVD esparso eficiente) |

## Dados Utilizados

| Arquivo | Conteúdo |
|---------|----------|
| `movie_encoding.tsv` | Features dos filmes (gêneros, idiomas, países) |
| `user_ratings.csv` | Avaliações dos usuários (USERID, MOVIEID, RATING) |
| `recommendations.tsv` | Histórico de recomendações anteriores (opcional) |
| `movie_category.tsv` | Categorias de diversidade para métricas de Commonality |

In [ ]:
!pip install numpy pandas scipy matplotlib tqdm --quiet

import numpy, pandas, scipy, matplotlib
print(f'numpy {numpy.__version__} | pandas {pandas.__version__} | scipy {scipy.__version__}')

In [ ]:
import os
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.sparse.linalg import svds
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print('Bibliotecas carregadas com sucesso.')

## 1. Upload dos Arquivos

Monte o Google Drive e configure os caminhos para os arquivos abaixo.

| Arquivo | Descrição | Obrigatório |
|---------|-----------|-------------|
| `movie_encoding.tsv` | Features dos filmes — gêneros, idiomas, países (gerado por `rs_movie_encoding.py`) | Sim |
| `user_ratings.csv` | Avaliações dos usuários: colunas `USERID`, `MOVIEID`, `RATING` | Sim |
| `recommendations.tsv` | Histórico de recomendações passadas: colunas `USERID`, `MOVIEID` | Não |
| `movie_category.tsv` | Categorias de diversidade: `director_women`, `director_nowhite`, etc. (gerado por `rs_movie_category.py`) | Para Commonality |

> **Dica**: coloque todos os arquivos em uma pasta no Google Drive (ex: `MinhaUnidade/rs_data/`) e ajuste `DRIVE_BASE` abaixo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configure os caminhos aqui ────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/rs_data/'

MOVIE_ENC_PATH = os.path.join(DRIVE_BASE, 'movie_encoding.tsv')
RATINGS_PATH   = os.path.join(DRIVE_BASE, 'user_ratings.csv')
RECS_PATH      = os.path.join(DRIVE_BASE, 'recommendations.tsv')   # opcional
CATEGORY_PATH  = os.path.join(DRIVE_BASE, 'movie_category.tsv')    # para Commonality

for label, path, required in [
    ('movie_encoding.tsv', MOVIE_ENC_PATH, True),
    ('user_ratings.csv',   RATINGS_PATH,   True),
    ('recommendations.tsv',RECS_PATH,      False),
    ('movie_category.tsv', CATEGORY_PATH,  False),
]:
    exists = os.path.exists(path)
    status = '✓' if exists else ('✗ OBRIGATÓRIO' if required else '— opcional')
    print(f'  {status}  {label}')

## 2. Carregamento e Exploração dos Dados

Antes de aplicar o SVD, é importante entender a estrutura dos dados:

- **Esparsidade**: diferentemente dos métodos de vizinhança, o SVD lida bem com alta esparsidade pois a imputação baseline preenche os valores ausentes antes da decomposição. Ainda assim, esparsidade muito alta (>99.9%) pode fazer a decomposição convergir para vieses ao invés de padrões de preferência.
- **Distribuição de ratings**: a média global $\mu$ e a variância influenciam diretamente a qualidade da imputação baseline e a convergência do SVD.
- **Tamanho da matriz**: a complexidade do SVD truncado é $O(n_{users} \times n_{items} \times k)$. Para matrizes muito grandes, `scipy.sparse.linalg.svds` usa o método de Krylov (ARPACK) que é eficiente mesmo para matrizes esparsas.

In [ ]:
# ── Carrega os dados ──────────────────────────────────────────────────────────
print('Carregando movie_encoding.tsv ...')
movies_df = pd.read_csv(MOVIE_ENC_PATH, sep='\t', low_memory=False)
genre_cols = [c for c in movies_df.columns if c.startswith('genre_')]
print(f'  {len(movies_df):,} filmes | {len(genre_cols)} gêneros')

print('Carregando user_ratings.csv ...')
ratings_df = pd.read_csv(RATINGS_PATH)
ratings_df.columns = ratings_df.columns.str.upper()
ratings_df = ratings_df.dropna(subset=['USERID','MOVIEID','RATING'])
ratings_df['USERID']  = ratings_df['USERID'].astype(int)
ratings_df['MOVIEID'] = ratings_df['MOVIEID'].astype(int)
ratings_df['RATING']  = ratings_df['RATING'].astype(float)
print(f'  {len(ratings_df):,} avaliações | {ratings_df["USERID"].nunique():,} usuários | {ratings_df["MOVIEID"].nunique():,} filmes')

if os.path.exists(RECS_PATH):
    recs_df = pd.read_csv(RECS_PATH, sep='\t')
    recs_df.columns = recs_df.columns.str.upper()
    print(f'Recomendações anteriores: {len(recs_df):,}')
else:
    recs_df = None

# ── Gráficos exploratórios ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploração dos Dados de Avaliações', fontsize=14, fontweight='bold')

# 1. Distribuição de ratings
ax = axes[0, 0]
ratings_df['RATING'].hist(bins=20, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribuição de Ratings')
ax.set_xlabel('Rating')
ax.set_ylabel('Frequência')
ax.axvline(ratings_df['RATING'].mean(), color='crimson', linestyle='--',
           label=f'Média global μ: {ratings_df["RATING"].mean():.2f}')
ax.legend()

# 2. Avaliações por usuário
ax = axes[0, 1]
n_per_user = ratings_df.groupby('USERID').size()
ax.hist(n_per_user, bins=50, color='darkorange', edgecolor='white', log=True)
ax.set_title('Avaliações por Usuário (escala log)')
ax.set_xlabel('Número de filmes avaliados')
ax.set_ylabel('Nº de usuários (log)')
ax.axvline(n_per_user.median(), color='crimson', linestyle='--',
           label=f'Mediana: {n_per_user.median():.0f}')
ax.legend()

# 3. Avaliações por filme
ax = axes[1, 0]
n_per_movie = ratings_df.groupby('MOVIEID').size()
ax.hist(n_per_movie, bins=50, color='seagreen', edgecolor='white', log=True)
ax.set_title('Avaliações por Filme (escala log)')
ax.set_xlabel('Número de avaliações')
ax.set_ylabel('Nº de filmes (log)')
ax.axvline(n_per_movie.median(), color='crimson', linestyle='--',
           label=f'Mediana: {n_per_movie.median():.0f}')
ax.legend()

# 4. Top gêneros
ax = axes[1, 1]
if genre_cols:
    genre_counts = movies_df[genre_cols].sum().sort_values(ascending=False).head(12)
    genre_counts.index = genre_counts.index.str.replace('genre_', '').str.replace('_', ' ').str.title()
    genre_counts.plot(kind='barh', ax=ax, color='mediumpurple')
    ax.set_title('Top-12 Gêneros no Catálogo')
    ax.set_xlabel('Número de filmes')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

n_users_total  = ratings_df['USERID'].nunique()
n_movies_total = ratings_df['MOVIEID'].nunique()
sparsity = 1 - len(ratings_df) / (n_users_total * n_movies_total)
print(f'\nEsparsidade da matriz: {sparsity:.4%}')
print(f'Tamanho da matriz R  : {n_users_total:,} × {n_movies_total:,} = {n_users_total*n_movies_total:,} entradas')

## 3. Divisão Treino / Teste por Usuário (80/20)

Para avaliar o algoritmo, dividimos as avaliações de **cada usuário individualmente** em 80% treino e 20% teste.

No contexto do SVD, essa divisão é importante por uma razão adicional: ao construir a matriz R, usamos **apenas os ratings de treino**. Os ratings de teste permanecem "escondidos" — o SVD não tem acesso a eles, e os usamos para verificar se as predições do modelo são corretas.

> ⚠️ **Atenção ao vazamento de dados (data leakage)**: o SVD é treinado sobre R_treino e a imputação baseline usa médias calculadas **somente sobre R_treino**. Usar dados do teste na imputação seria data leakage e inflaria artificialmente as métricas.

**Parâmetros configuráveis:**
- `TEST_RATIO`: proporção do teste (padrão: 0.2 = 20%).
- `MIN_RATINGS_SPLIT`: usuários com menos que este número ficam inteiramente no treino.
- `RELEVANCE_THRESHOLD`: rating mínimo para considerar um filme como "relevante".

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
TEST_RATIO          = 0.20
MIN_RATINGS_SPLIT   = 5
RELEVANCE_THRESHOLD = 4.0

# ── Split por usuário ─────────────────────────────────────────────────────────
train_ratings: dict[int, dict[int, float]] = defaultdict(dict)
test_ratings:  dict[int, dict[int, float]] = defaultdict(dict)

for uid, group in ratings_df.groupby('USERID'):
    rows = group[['MOVIEID', 'RATING']].values.tolist()
    random.shuffle(rows)

    if len(rows) < MIN_RATINGS_SPLIT:
        for mid, rat in rows:
            train_ratings[uid][int(mid)] = float(rat)
    else:
        n_test = max(1, int(len(rows) * TEST_RATIO))
        for mid, rat in rows[n_test:]:
            train_ratings[uid][int(mid)] = float(rat)
        for mid, rat in rows[:n_test]:
            test_ratings[uid][int(mid)]  = float(rat)

train_ratings = dict(train_ratings)
test_ratings  = dict(test_ratings)

train_users      = [u for u in train_ratings if len(train_ratings[u]) >= MIN_RATINGS_SPLIT]
all_movies_train = sorted(set(m for u in train_ratings.values() for m in u))

n_test_users = sum(1 for u in train_users if u in test_ratings and
                   any(r >= RELEVANCE_THRESHOLD for r in test_ratings[u].values()))

MIN_RATING = float(ratings_df['RATING'].min())
MAX_RATING = float(ratings_df['RATING'].max())

print(f'Usuários totais      : {ratings_df["USERID"].nunique():,}')
print(f'Usuários no treino   : {len(train_users):,}')
print(f'Usuários com teste   : {len(test_ratings):,}')
print(f'  (com item relevante): {n_test_users:,}')
print(f'Filmes no catálogo   : {len(all_movies_train):,}')
print(f'Rating range         : [{MIN_RATING}, {MAX_RATING}]')
print(f'\nRatings no treino    : {sum(len(v) for v in train_ratings.values()):,}')
print(f'Ratings no teste     : {sum(len(v) for v in test_ratings.values()):,}')

## 4. Fatoração de Matriz via SVD Truncado

### Pipeline de treinamento

**Passo 1 — Construção da matriz R** ($n_{users} \times n_{items}$):
- $R[u, i] = r_{u,i}$ se o usuário $u$ avaliou o filme $i$ no treino
- $R[u, i] = $ NaN caso contrário

**Passo 2 — Cálculo dos vieses (baseline predictor)**:
$$\mu = \text{média global de todos os ratings}$$
$$b_u = \bar{r}_u - \mu \quad \text{(viés do usuário)}$$
$$b_i = \bar{r}_i - \mu \quad \text{(viés do item)}$$

**Passo 3 — Imputação dos valores ausentes**:
$$R_{\text{imputado}}[u, i] = \begin{cases} r_{u,i} & \text{se observado} \\ \mu + b_u + b_i & \text{se ausente} \end{cases}$$

**Passo 4 — Centralização** (SVD funciona melhor em dados centrados):
$$R_{\text{centrado}} = R_{\text{imputado}} - \mu$$

**Passo 5 — SVD Truncado com $k$ fatores** (via `scipy.sparse.linalg.svds`):
$$R_{\text{centrado}} \approx U_k \Sigma_k V_k^\top$$

**Passo 6 — Reconstrução e clipping**:
$$\hat{R} = \text{clip}\left(U_k \Sigma_k V_k^\top + \mu,\; r_{\min},\; r_{\max}\right)$$

### Escolha de k

O número de fatores latentes $k$ é o hiperparâmetro mais importante. O gráfico de **variância explicada** pelos valores singulares ajuda a escolher $k$:
- Procure o **joelho da curva** — ponto após o qual adicionar mais fatores traz ganho marginal.
- Valores típicos: 10–200 dependendo do tamanho do dataset.
- $k$ muito pequeno → underfitting; $k$ muito grande → overfitting (memoriza o ruído da imputação).

> `scipy.sparse.linalg.svds` usa o algoritmo ARPACK (Krylov) — não precisa da matriz completa em memória, é eficiente para matrizes grandes.

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
N_FACTORS = 50   # número de fatores latentes k

# ── Índices para a matriz ─────────────────────────────────────────────────────
train_users_list = list(train_users)
user_to_idx  = {u: i for i, u in enumerate(train_users_list)}
movie_to_idx = {m: j for j, m in enumerate(all_movies_train)}

n_users_svd  = len(train_users_list)
n_movies_svd = len(all_movies_train)

# ── Passo 1: constrói a matriz R ──────────────────────────────────────────────
print(f'Passo 1 — Construindo R: {n_users_svd:,} usuários × {n_movies_svd:,} filmes ...')
R = np.full((n_users_svd, n_movies_svd), np.nan, dtype=np.float32)

for uid, movies in train_ratings.items():
    i = user_to_idx.get(uid)
    if i is None:
        continue
    for mid, rat in movies.items():
        j = movie_to_idx.get(mid)
        if j is not None:
            R[i, j] = rat

# ── Passo 2: calcula vieses ───────────────────────────────────────────────────
print('Passo 2 — Calculando vieses (μ, b_u, b_i) ...')
global_mean = float(np.nanmean(R))
user_bias   = np.nan_to_num(np.nanmean(R, axis=1) - global_mean, nan=0.0)  # (n_users,)
item_bias   = np.nan_to_num(np.nanmean(R, axis=0) - global_mean, nan=0.0)  # (n_movies,)

print(f'  Média global μ : {global_mean:.4f}')
print(f'  Viés de usuário: min={user_bias.min():.3f}  max={user_bias.max():.3f}')
print(f'  Viés de item   : min={item_bias.min():.3f}  max={item_bias.max():.3f}')

# ── Passo 3: imputação baseline ───────────────────────────────────────────────
print('Passo 3 — Imputando valores ausentes com baseline predictor ...')
baseline = global_mean + user_bias[:, None] + item_bias[None, :]  # broadcast (n_users, n_movies)
R_filled = np.where(np.isnan(R), baseline, R).astype(np.float64)

n_observed = int((~np.isnan(R)).sum())
n_imputed  = n_users_svd * n_movies_svd - n_observed
print(f'  Valores observados : {n_observed:,}')
print(f'  Valores imputados  : {n_imputed:,}  ({n_imputed/(n_users_svd*n_movies_svd):.2%})')

# Libera R original (não mais necessário)
del R

# ── Passo 4: centralização ────────────────────────────────────────────────────
print('Passo 4 — Centralizando pela média global ...')
R_centered = R_filled - global_mean
del R_filled

# ── Passo 5: SVD truncado ─────────────────────────────────────────────────────
print(f'Passo 5 — SVD truncado com k={N_FACTORS} fatores latentes ...')
# svds retorna os k menores valores singulares por padrão → pede k maiores via which='LM'
U, sigma, Vt = svds(R_centered, k=N_FACTORS, which='LM')
del R_centered

# svds retorna em ordem crescente → reverter para decrescente
sort_idx = np.argsort(sigma)[::-1]
U     = U[:, sort_idx]      # (n_users, k)
sigma = sigma[sort_idx]      # (k,)
Vt    = Vt[sort_idx, :]      # (k, n_movies)

variance_explained = np.cumsum(sigma**2) / np.sum(sigma**2) * 100
print(f'  Top-5 valores singulares: {sigma[:5].round(2)}')
print(f'  Variância explicada por k={N_FACTORS}: {variance_explained[-1]:.1f}%')

# ── Passo 6: reconstrói R_hat e faz clipping ─────────────────────────────────
print('Passo 6 — Reconstruindo R̂ e aplicando clipping ...')
R_hat = U @ np.diag(sigma) @ Vt + global_mean   # (n_users, n_movies)
R_hat = np.clip(R_hat, MIN_RATING, MAX_RATING).astype(np.float32)

print(f'\nSVD concluído!')
print(f'  Shape de U    : {U.shape}   (usuários × fatores)')
print(f'  Shape de Vt   : {Vt.shape}  (fatores × filmes)')
print(f'  Shape de R̂   : {R_hat.shape}')
print(f'  Score médio predito: {R_hat.mean():.3f}')

# ── Gráfico de variância explicada ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, N_FACTORS+1), variance_explained, 'o-', color='steelblue',
        markersize=4, label='Variância acumulada')
ax.bar(range(1, N_FACTORS+1), (sigma**2 / sigma**2.sum() * 100),
       color='lightsteelblue', alpha=0.5, label='Variância por fator')
for threshold in [50, 80, 90]:
    ax.axhline(threshold, color='crimson', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.text(N_FACTORS * 0.98, threshold + 1, f'{threshold}%', color='crimson',
            ha='right', fontsize=8)
ax.set_xlabel('Número de fatores latentes (k)')
ax.set_ylabel('Variância explicada (%)')
ax.set_title(f'Variância Explicada pelos {N_FACTORS} Fatores Latentes (SVD)')
ax.legend()
ax.set_xlim(0, N_FACTORS + 1)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 5. Extração de Scores

Com a matriz reconstruída $\hat{R}$, obter o score predito para qualquer par (usuário, filme) é trivial:

$$\hat{r}_{u,i} = \hat{R}[\text{idx}(u),\; \text{idx}(i)]$$

Esta é a principal vantagem do SVD sobre os métodos de vizinhança: **não há custo de inferência** além de um simples lookup matricial. Todo o custo computacional foi pago uma única vez durante a decomposição.

Para cada usuário, extraímos os scores apenas dos **filmes não avaliados** no treino — estes são os candidatos a recomendação.

> **Interpretação do score**: diferente dos métodos de vizinhança (onde o score é uma predição de rating na escala original), o score aqui é a predição de $\hat{r}_{u,i}$ após clipping. Valores próximos de `MAX_RATING` indicam alta preferência estimada; valores próximos de `MIN_RATING` indicam baixa preferência.

In [ ]:
# ── Extrai scores para filmes não avaliados por cada usuário ─────────────────
user_scores: dict[int, dict[int, float]] = {}

for uid in tqdm(train_users_list, desc='Extraindo scores'):
    i = user_to_idx[uid]

    # Índices dos filmes já avaliados pelo usuário
    rated_indices = np.array(
        [movie_to_idx[m] for m in train_ratings[uid] if m in movie_to_idx],
        dtype=np.int32
    )

    # Máscara de filmes não avaliados
    mask = np.ones(n_movies_svd, dtype=bool)
    if rated_indices.size > 0:
        mask[rated_indices] = False

    unrated_idx  = np.where(mask)[0]
    scores_arr   = R_hat[i, unrated_idx]                     # (n_unrated,)

    user_scores[uid] = {
        all_movies_train[j]: float(scores_arr[k])
        for k, j in enumerate(unrated_idx)
    }

# Resumo
n_scored            = sum(len(s) for s in user_scores.values())
n_users_with_scores = sum(1 for s in user_scores.values() if s)
all_score_vals      = [s for sc in user_scores.values() for s in sc.values()]

print(f'Scores extraídos!')
print(f'  Usuários com ao menos 1 score : {n_users_with_scores:,}')
print(f'  Total de (usuário, filme) com score: {n_scored:,}')
print(f'  Score médio : {np.mean(all_score_vals):.3f}')
print(f'  Score mín   : {np.min(all_score_vals):.3f}')
print(f'  Score máx   : {np.max(all_score_vals):.3f}')

## 6. Geração do Ranking de Recomendações

Com os scores calculados, geramos duas estratégias de ranking:

### Top-N Determinístico
Ordena todos os filmes com score pelo valor decrescente e retorna os **N primeiros**. Maximiza a relevância esperada mas produz recomendações estáticas.

### Top-N Estocástico
Converte os scores em **probabilidades** via softmax e amostra N filmes **sem reposição**:

$$p(i) = \frac{e^{\hat{r}_{u,i} / T}}{\sum_{j} e^{\hat{r}_{u,j} / T}}$$

O parâmetro de **temperatura** $T$ controla a aleatoriedade:
- $T \to 0$: equivale a Top-N determinístico.
- $T = 1$: proporcional ao score predito.
- $T \to \infty$: completamente aleatório.

No SVD, scores muito similares entre itens são comuns (o espaço latente é contínuo). O estocástico com temperatura adequada introduz **diversidade** sem perder demasiada qualidade.

**Parâmetros:**
- `RANKING_METHOD`: `'topn'` ou `'stochastic'`
- `N_RECOMMENDATIONS`: número de filmes por recomendação
- `MIN_SCORE_STOCHASTIC`: score mínimo para entrar no sorteio estocástico
- `TEMPERATURE`: temperatura do softmax

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
RANKING_METHOD       = 'topn'   # 'topn' | 'stochastic'
N_RECOMMENDATIONS    = 10
MIN_SCORE_STOCHASTIC = 3.0
TEMPERATURE          = 1.0

# ── Funções de ranking ────────────────────────────────────────────────────────

def top_n_ranking(scores: dict, n: int, exclude: set) -> list:
    filtered = {m: s for m, s in scores.items() if m not in exclude}
    return sorted(filtered, key=filtered.get, reverse=True)[:n]


def stochastic_ranking(scores: dict, n: int, min_score: float,
                        temperature: float, exclude: set) -> list:
    eligible = {m: s for m, s in scores.items()
                if s >= min_score and m not in exclude}
    if not eligible:
        return top_n_ranking(scores, n, exclude)

    movies = list(eligible.keys())
    raw_s  = np.array([eligible[m] for m in movies], dtype=np.float64)
    logits = raw_s / max(temperature, 1e-8)
    exp_s  = np.exp(logits - logits.max())
    probs  = exp_s / exp_s.sum()

    k = min(n, len(movies))
    chosen = np.random.choice(len(movies), size=k, replace=False, p=probs)
    return [movies[i] for i in chosen]


# ── Gera rankings ─────────────────────────────────────────────────────────────
user_rankings: dict[int, list] = {}

for uid in tqdm(train_users_list, desc='Gerando rankings'):
    scores  = user_scores.get(uid, {})
    exclude = set(train_ratings[uid].keys())

    if RANKING_METHOD == 'stochastic':
        ranking = stochastic_ranking(scores, N_RECOMMENDATIONS,
                                      MIN_SCORE_STOCHASTIC, TEMPERATURE, exclude)
    else:
        ranking = top_n_ranking(scores, N_RECOMMENDATIONS, exclude)

    user_rankings[uid] = ranking

lens = [len(r) for r in user_rankings.values()]
print(f'Rankings gerados: {len(user_rankings):,} usuários')
print(f'  Método        : {RANKING_METHOD.upper()}')
print(f'  Tamanho médio : {np.mean(lens):.1f} filmes')
print(f'  Com ranking   : {sum(1 for l in lens if l > 0):,} usuários')

## 7. Métricas de Avaliação

Avaliamos a qualidade do ranking com as seguintes métricas, implementadas em `rs_metrics.py`:

| Métrica | O que mede | Fórmula resumida |
|---------|------------|------------------|
| **Precision@K** | De K recomendações, quantas são relevantes? | `|top-K ∩ relevantes| / K` |
| **Recall@K** | De todos os relevantes, quantos estão no top-K? | `|top-K ∩ relevantes| / |relevantes|` |
| **F1@K** | Equilíbrio entre Precision e Recall | `2·P·R / (P+R)` |
| **NDCG@K** | Qualidade do ranking ponderando pela posição | Ganho descontado normalizado |
| **MAP** | Precisão média ponderada pela posição do primeiro acerto | Média dos Average Precisions |
| **Hit Rate** | % de usuários com ao menos 1 item relevante | `usuários_com_hit / total` |
| **MRR** | Posição média do primeiro item relevante | `média(1/posição_do_primeiro_hit)` |
| **Coverage** | % do catálogo que o sistema recomenda | `|filmes_únicos_recomendados| / |catálogo|` |

**Itens relevantes**: filmes no conjunto de teste com `RATING >= RELEVANCE_THRESHOLD`.

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/rs_data/')
from rs_metrics import RankingMetrics

K_METRICS = N_RECOMMENDATIONS

rankings_eval = []
relevant_eval = []
eval_user_ids = []

for uid in train_users_list:
    if uid not in user_rankings or not user_rankings[uid]:
        continue
    if uid not in test_ratings:
        continue

    relevant = [m for m, r in test_ratings[uid].items() if r >= RELEVANCE_THRESHOLD]
    if not relevant:
        continue

    rankings_eval.append(user_rankings[uid])
    relevant_eval.append(relevant)
    eval_user_ids.append(uid)

print(f'Usuários avaliados: {len(eval_user_ids):,}')

m = RankingMetrics()

prec  = np.mean([m.precision_at_k(r, rel, K_METRICS) for r, rel in zip(rankings_eval, relevant_eval)])
rec   = np.mean([m.recall_at_k(r, rel, K_METRICS)    for r, rel in zip(rankings_eval, relevant_eval)])
ndcg  = np.mean([m.ndcg_at_k(r, rel, K_METRICS)      for r, rel in zip(rankings_eval, relevant_eval)])
f1    = m.f1_score(prec, rec)
map_s = m.mean_average_precision(rankings_eval, relevant_eval)
hr    = m.hit_rate(rankings_eval, relevant_eval)
mrr   = m.mean_reciprocal_rank(rankings_eval, relevant_eval)
cov   = m.coverage(rankings_eval, catalog_size=len(all_movies_train))

metric_results = {
    f'Precision@{K_METRICS}': prec,
    f'Recall@{K_METRICS}':    rec,
    f'F1@{K_METRICS}':        f1,
    f'NDCG@{K_METRICS}':      ndcg,
    'MAP':                     map_s,
    'Hit Rate':                hr,
    'MRR':                     mrr,
    'Coverage':                cov,
}

print(f'\n{"Métrica":<20}  {"Valor":>8}')
print('─' * 32)
for name, val in metric_results.items():
    print(f'{name:<20}  {val:>8.4f}')

## 8. Métrica de Commonality (Diversidade de Exposição)

A métrica **Commonality**, implementada em `rs_commonality.py`, mede a probabilidade de que **todos os usuários simultaneamente** se familiarizem com uma categoria de filmes.

### Conceitos

**Probabilidade de Browsing** (RBP — *Rank-Biased Precision*):
$$\Pr(k) = (1 - \gamma) \cdot \gamma^{k-1}$$

**Familiaridade** do usuário $u$ com a categoria $g$:
$$\Pr(F_{u,g} | \pi_u) = \sum_{k=1}^{N} \Pr(k) \cdot R(\pi_u, k, g)$$

**Commonality** para todos os usuários:
$$C_g(\pi) = \prod_{u} \Pr(F_{u,g} | \pi_u)$$

### Interpretação no contexto do SVD

O SVD otimiza a predição de rating sem considerar diversidade. A Commonality permite verificar se o modelo, ao priorizar filmes com alto score predito, acaba reforçando vieses do catálogo: por exemplo, se filmes de diretores homens brancos americanos tendem a ter mais avaliações (e portanto mais influência no SVD), o sistema pode sistematicamente subrecomendado categorias de diversidade.

### Categorias de Diversidade

| Categoria | Critério |
|-----------|----------|
| Diretoras Mulheres | `director_women = 1` |
| Dir. Raça Não-Branca | `director_nowhite = 1` |
| Dir. Região Não-EU/NA | `director_region = 1` |
| Origem Não-EU/NA | `movie_region = 1` |
| Produção Brasileira | `movie_region_bra = 1` |

In [ ]:
from rs_commonality import CommonalityCalculator

GAMMA              = 0.8
COMMONALITY_SAMPLE = 200

if not os.path.exists(CATEGORY_PATH):
    raise FileNotFoundError(
        f'Arquivo não encontrado: {CATEGORY_PATH}\n'
        'Execute rs_movie_category.py para gerar movie_category.tsv.'
    )

cat_df = pd.read_csv(CATEGORY_PATH, sep='\t')
print(f'Categorias carregadas: {len(cat_df):,} filmes')

DIVERSITY_CATEGORIES = {
    'Diretoras Mulheres':     set(cat_df[cat_df['director_women']   == 1]['movieid']),
    'Dir. Raça Não-Branca':   set(cat_df[cat_df['director_nowhite'] == 1]['movieid']),
    'Dir. Reg. Não-EU/NA':    set(cat_df[cat_df['director_region']  == 1]['movieid']),
    'Origem Não-EU/NA':       set(cat_df[cat_df['movie_region']     == 1]['movieid']),
    'Produção Brasileira':    set(cat_df[cat_df['movie_region_bra'] == 1]['movieid']),
}

for cat, items in DIVERSITY_CATEGORIES.items():
    print(f'  {cat:<26}: {len(items):,} filmes')

eligible_users  = [u for u in train_users_list if user_rankings.get(u)]
sample_size     = min(COMMONALITY_SAMPLE, len(eligible_users))
sample_uids     = random.sample(eligible_users, sample_size)
sample_rankings = [user_rankings[u] for u in sample_uids]

print(f'\nAmostra de {sample_size} usuários para Commonality (γ={GAMMA})')

calc = CommonalityCalculator(gamma=GAMMA)

commonality_results = {}
print(f'\n{"Categoria":<26}  {"Fam. Média":>10}  {"Fam. Std":>9}  {"Commonality":>12}')
print('─' * 65)

for cat_name, cat_items in DIVERSITY_CATEGORIES.items():
    cat_list = list(cat_items)

    familiarities = [calc.familiarity(ranking, cat_list) for ranking in sample_rankings]
    commonality   = calc.categoria_commonality(sample_rankings, cat_list)

    commonality_results[cat_name] = {
        'fam_mean':    float(np.mean(familiarities)),
        'fam_std':     float(np.std(familiarities)),
        'commonality': float(commonality),
        'n_items':     len(cat_items),
    }

    fam_m = commonality_results[cat_name]['fam_mean']
    fam_s = commonality_results[cat_name]['fam_std']
    comm  = commonality_results[cat_name]['commonality']
    print(f'{cat_name:<26}  {fam_m:>10.4f}  {fam_s:>9.4f}  {comm:>12.6f}')

print(f'\nNota: Commonality é o produto das familiaridades — decresce com mais usuários.')

## 9. Visualização dos Resultados

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f'Resultados — MF-SVD  |  Método: {RANKING_METHOD.upper()}  |  '
    f'Top-{N_RECOMMENDATIONS}  |  k={N_FACTORS} fatores latentes',
    fontsize=13, fontweight='bold'
)

# ── 1. Métricas de avaliação ──────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
metric_names = list(metric_results.keys())
metric_vals  = list(metric_results.values())
colors_m = ['steelblue' if v > 0.1 else 'lightsteelblue' for v in metric_vals]
bars = ax1.barh(metric_names, metric_vals, color=colors_m, edgecolor='white')
ax1.set_xlim(0, max(metric_vals) * 1.25)
ax1.set_title('Métricas de Avaliação', fontweight='bold')
ax1.set_xlabel('Valor')
for bar, val in zip(bars, metric_vals):
    ax1.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', fontsize=8)
ax1.invert_yaxis()

# ── 2. Familiaridade média por categoria ─────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
cat_names = list(commonality_results.keys())
fam_means = [commonality_results[c]['fam_mean'] for c in cat_names]
fam_stds  = [commonality_results[c]['fam_std']  for c in cat_names]
y_pos = list(range(len(cat_names)))
ax2.barh(y_pos, fam_means, xerr=fam_stds, color='darkorange',
         edgecolor='white', capsize=4, alpha=0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(cat_names, fontsize=9)
ax2.set_title('Familiaridade Média por Categoria\n(com desvio padrão)', fontweight='bold')
ax2.set_xlabel('Familiaridade esperada')
ax2.invert_yaxis()

# ── 3. Commonality por categoria ──────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
comm_vals = [commonality_results[c]['commonality'] for c in cat_names]
ax3.barh(y_pos, comm_vals, color='seagreen', edgecolor='white', alpha=0.8)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(cat_names, fontsize=9)
ax3.set_title(f'Commonality por Categoria\n(amostra {sample_size} usuários)', fontweight='bold')
ax3.set_xlabel('Commonality (produto das familiaridades)')
ax3.xaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax3.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
ax3.invert_yaxis()

# ── 4. Valores singulares e variância explicada ───────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)
variance_explained = np.cumsum(sigma**2) / np.sum(sigma**2) * 100
ax4.plot(range(1, N_FACTORS + 1), variance_explained, 'o-',
         color='steelblue', markersize=3, linewidth=1.5)
ax4.fill_between(range(1, N_FACTORS + 1), variance_explained, alpha=0.15, color='steelblue')
for threshold in [50, 80, 90]:
    if variance_explained[-1] >= threshold:
        cross_k = np.searchsorted(variance_explained, threshold) + 1
        ax4.axhline(threshold, color='crimson', linestyle='--', alpha=0.6, linewidth=0.8)
        ax4.text(N_FACTORS * 0.98, threshold + 1.5, f'{threshold}% (k={cross_k})',
                 color='crimson', ha='right', fontsize=7)
ax4.set_title(f'Variância Explicada (k={N_FACTORS} fatores)', fontweight='bold')
ax4.set_xlabel('Fatores latentes')
ax4.set_ylabel('Variância acumulada (%)')
ax4.set_ylim(0, 105)

# ── 5. Distribuição dos scores preditos ───────────────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)
sample_scores = np.random.choice(all_score_vals, min(50000, len(all_score_vals)), replace=False)
ax5.hist(sample_scores, bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
ax5.axvline(np.mean(sample_scores), color='crimson', linestyle='--',
            label=f'Média: {np.mean(sample_scores):.2f}')
ax5.axvline(global_mean, color='orange', linestyle=':',
            label=f'Média global: {global_mean:.2f}')
ax5.axvline(MIN_SCORE_STOCHASTIC, color='green', linestyle=':',
            label=f'Min estocástico: {MIN_SCORE_STOCHASTIC}')
ax5.set_title('Distribuição de Scores Preditos', fontweight='bold')
ax5.set_xlabel('Score predito $\\hat{r}_{u,i}$')
ax5.set_ylabel('Frequência')
ax5.legend(fontsize=7)

# ── 6. Tamanho das categorias de diversidade ──────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
n_items_cat   = [commonality_results[c]['n_items'] for c in cat_names]
total_catalog = len(all_movies_train)
pcts = [v / total_catalog * 100 if total_catalog > 0 else 0 for v in n_items_cat]
ax6.barh(y_pos, pcts, color='steelblue', edgecolor='white', alpha=0.8)
ax6.set_yticks(y_pos)
ax6.set_yticklabels(cat_names, fontsize=9)
ax6.set_title(f'Tamanho das Categorias\n(% do catálogo — {total_catalog:,} filmes)', fontweight='bold')
ax6.set_xlabel('% do catálogo')
for i, (pct, n) in enumerate(zip(pcts, n_items_cat)):
    ax6.text(pct + 0.3, i, f'{n:,} ({pct:.1f}%)', va='center', fontsize=8)
ax6.invert_yaxis()

plt.tight_layout()
plt.savefig('resultados_mf_svd.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em resultados_mf_svd.png')